# Handling Missing Values

Real-world data is often messy and contains missing values. Handling them correctly is a crucial preprocessing step, as most machine learning algorithms cannot handle missing data directly.

## Understanding Mechanisms of Missing Data

Before deciding *how* to handle missing data, it's essential to understand *why* it's missing. There are three main mechanisms:

1. **Missing Completely at Random (MCAR)**: The probability of a value being missing is the same for all observations. There's no relationship between whether a data point is missing and any values in the dataset. (e.g., A sensor randomly dropped a packet).
2. **Missing at Random (MAR)**: The probability of a value being missing depends on other observed variables, but not on the missing value itself. (e.g., Men might be less likely to fill out a survey question about depression than women, but it doesn't depend on their actual depression level).
3. **Missing Not at Random (MNAR)**: The probability of missingness depends on the unobserved value itself. (e.g., People with very high incomes might be less likely to report their income).

Identifying the mechanism helps choose the right strategy: MCAR allows for simple deletion, while MAR and MNAR require careful imputation or specialized modeling.

## 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

## 2. Loading the Dataset

We will use the Titanic dataset which contains several missing values.

In [ ]:
# Assuming the notebook is run from the 01_Preprocessing directory
df = pd.read_csv('../titanic.csv')
print(df.head())

## 3. Identifying Missing Values

In [ ]:
# Check total missing values per column
print(df.isnull().sum())

# Percentage of missing values
missing_percentages = (df.isnull().sum() / len(df)) * 100
print(missing_percentages)

In [ ]:
# Visualizing missing data percentage
plt.figure(figsize=(10, 5))
missing_percentages[missing_percentages > 0].sort_values(ascending=False).plot(kind='bar', color='coral')
plt.title('Percentage of Missing Values per Feature')
plt.ylabel('Percentage (%)')
plt.show()

In [ ]:
# Visualizing missing data map (where yellow lines represent missing data)
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Data Map')
plt.show()

## 4. Deletion Strategies

- **Listwise Deletion (Dropping Rows)**: Dropping rows with any missing values. Only recommended if data is MCAR and you have a large dataset, as it can lead to massive information loss.
- **Column Deletion**: Dropping columns with a very high percentage of missing values (e.g., >70% missing).

In [ ]:
# Dropping rows with any missing value
df_dropna_rows = df.dropna()
print(f'Original shape: {df.shape}, Shape after dropping rows: {df_dropna_rows.shape}')

# Dropping columns with more than 70% missing values
threshold = len(df) * 0.7
df_dropna_cols = df.dropna(axis=1, thresh=threshold)
print(f'Shape after dropping mostly empty columns: {df_dropna_cols.shape}')

## 5. Basic Imputation

Filling missing values with a central tendency measure (mean, median, mode) or a constant value.

**When to use which?**
- **Mean**: Use for continuous, normally distributed data without extreme outliers.
- **Median**: Use for continuous data that is skewed or has outliers (median is robust to outliers).
- **Mode**: Use for categorical or discrete data.

In [ ]:
# Using pandas
df_filled_pandas = df.copy()
# Age is slightly skewed, median is a safe choice
df_filled_pandas['Age'].fillna(df_filled_pandas['Age'].median(), inplace=True)
# Embarked is categorical, we use the mode (most frequent value)
df_filled_pandas['Embarked'].fillna(df_filled_pandas['Embarked'].mode()[0], inplace=True)

print('Missing after pandas fill:\n', df_filled_pandas[['Age', 'Embarked']].isnull().sum())

In [ ]:
# Using scikit-learn SimpleImputer
imputer_num = SimpleImputer(strategy='median')
imputer_cat = SimpleImputer(strategy='most_frequent')

df_sklearn = df.copy()
df_sklearn[['Age', 'Fare']] = imputer_num.fit_transform(df_sklearn[['Age', 'Fare']])
df_sklearn[['Embarked']] = imputer_cat.fit_transform(df_sklearn[['Embarked']])

print('Missing after sklearn fill:\n', df_sklearn[['Age', 'Fare', 'Embarked']].isnull().sum())

## 6. Advanced Imputation

Using more sophisticated methods that take relationships between features into account.

**K-Nearest Neighbors (KNN) Imputation:**
Finds the 'K' most similar samples (neighbors) in the dataset and averages their values to fill the missing entry. Requires scaling the data first in practice!

In [ ]:
# KNN Imputer
knn_imputer = KNNImputer(n_neighbors=5)
df_knn = df[['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']].copy()
df_knn_imputed = pd.DataFrame(knn_imputer.fit_transform(df_knn), columns=df_knn.columns)

print('Missing after KNN imputation:\n', df_knn_imputed.isnull().sum())

**Iterative Imputation (MICE - Multiple Imputation by Chained Equations):**
Models each feature with missing values as a function of other features, iteratively predicting the missing values. Often yields the best results.

In [ ]:
# Iterative Imputer (MICE)
mice_imputer = IterativeImputer(max_iter=10, random_state=42)
df_mice = df[['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']].copy()
df_mice_imputed = pd.DataFrame(mice_imputer.fit_transform(df_mice), columns=df_mice.columns)

print('Missing after MICE imputation:\n', df_mice_imputed.isnull().sum())

## Summary

- **Deletion** is acceptable if missingness is completely random (MCAR) and the dataset is large.
- **Basic Imputation (Mean/Median/Mode)** is fast and simple but can artificially reduce variance and ignore relationships between features.
- **Advanced Imputation (KNN/Iterative)** is more accurate as it leverages feature correlations, but it is computationally expensive and can be sensitive to outliers.